# NLP Experiment 6 – Text generation with RNN, LSTM and GRU (Shakespeare)
**Runtime:** Python 3 + **T4 GPU** (Runtime → Change runtime type). Runtime → Run all. (CPU works, ~5× slower.)

**Character-level language model:** given the characters so far, predict the next character.
Train on Shakespeare, then generate new text from a seed sentence. Compare SimpleRNN vs LSTM vs GRU.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## (a) Load and preprocess the text

In [ ]:
path = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")
text = open(path, encoding="utf-8").read().lower()
text = text[:300000]                   # first 300k characters, enough for a demo
print("Characters:", len(text))
print(text[:250])

In [ ]:
chars = sorted(set(text))                              # vocabulary of characters
char_to_index = {c: i for i, c in enumerate(chars)}
index_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)
print("Vocabulary size:", vocab_size)

encoded = np.array([char_to_index[c] for c in text])
print(text[:10], "->", encoded[:10])

## Prepare training sequences
Input = 40 characters, target = the **same 40 characters shifted by one** (the next character at every step).
`"to be or no"` → `"o be or not"`.

In [ ]:
seq_length = 40
step = 3
X, y = [], []
for i in range(0, len(encoded) - seq_length - 1, step):
    X.append(encoded[i : i + seq_length])
    y.append(encoded[i + 1 : i + seq_length + 1])
X = np.array(X)
y = np.array(y)
print("Sequences:", X.shape, y.shape)

## (b) Build and train a SimpleRNN

In [ ]:
def build_model(rnn_layer):
    model = Sequential([
        Embedding(vocab_size, 64),                 # character id -> 64-number vector
        rnn_layer,                                 # SimpleRNN / LSTM / GRU with 128 units
        Dense(vocab_size, activation="softmax"),   # probability of each next character
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
    return model

rnn_model = build_model(SimpleRNN(128, return_sequences=True))
history_rnn = rnn_model.fit(X, y, batch_size=128, epochs=5, validation_split=0.1)

## Generate text from a seed sentence

In [ ]:
def generate(model, seed, length=300, temperature=0.7):
    result = seed.lower()
    for _ in range(length):
        last = result[-seq_length:]
        x = np.array([[char_to_index.get(c, 0) for c in last]])
        probs = model.predict(x, verbose=0)[0, -1]            # prediction for the next character
        probs = np.log(probs + 1e-8) / temperature             # temperature: <1 safer, >1 more random
        probs = np.exp(probs) / np.sum(np.exp(probs))
        next_index = np.random.choice(vocab_size, p=probs)
        result += index_to_char[next_index]
    return result

print(generate(rnn_model, "to be or not to be"))

## (c) Variations: LSTM and GRU

In [ ]:
lstm_model = build_model(LSTM(128, return_sequences=True))
history_lstm = lstm_model.fit(X, y, batch_size=128, epochs=5, validation_split=0.1)

gru_model = build_model(GRU(128, return_sequences=True))
history_gru = gru_model.fit(X, y, batch_size=128, epochs=5, validation_split=0.1)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_rnn.history["val_loss"], label="SimpleRNN")
plt.plot(history_lstm.history["val_loss"], label="LSTM")
plt.plot(history_gru.history["val_loss"], label="GRU")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("RNN vs LSTM vs GRU")
plt.legend()
plt.show()

for name, h in [("SimpleRNN", history_rnn), ("LSTM", history_lstm), ("GRU", history_gru)]:
    print(f"{name:10} final validation loss: {h.history['val_loss'][-1]:.3f}")

In [ ]:
seed = "love is"
for name, model in [("SimpleRNN", rnn_model), ("LSTM", lstm_model), ("GRU", gru_model)]:
    print("=" * 20, name)
    print(generate(model, seed, length=200))